#Ensamble Heterogéneo por Soft Voting (Celda por Celda)
##1.Entrenamiento del Modelo 1 (xlm-roberta-large Sin Aumentación) y Persistencia de Probabilidades
###¿Qué se hizo?

* Se liberó explícitamente la memoria VRAM residual de la ejecución previa (gc.collect(), torch.cuda.empty_cache(), limpieza de sys.last_traceback).

* Se instanció y entrenó desde el checkpoint preentrenado xlm-roberta-large durante 4 épocas (AdamW, LR = 1e-5, cosine schedule con 10% de warmup, recorte de gradiente 1.0) utilizando exclusivamente los datos reales originales (trn_df, 17,055 registros) sin ningún tipo de aumentación sintética.

* Para mantener la compatibilidad de memoria en GPU T4/P100 sin alterar la matemática del entrenamiento, se activó gradient checkpointing (model_1.gradient_checkpointing_enable()), micro-lote batch_size = 8 y acumulación de gradientes cada 2 pasos (tamaño de lote efectivo de 16), con longitud máxima max_len = 160.

* Al finalizar cada época se evaluó val_df (3,411 registros intactos) y se almacenaron tanto los pesos óptimos (best_xlmr_sin_aug.pt, con Val Accuracy = 0.9062) como las matrices completas de probabilidad posterior (softmax) sobre validación (val_probs_xlmr_sin_aug.npy) y evaluación (eval_probs_xlmr_sin_aug.npy).

###¿Por qué se decidió hacer eso?

* Conservar nuestro mejor predictor individual sobre lenguaje natural del dominio: En el estudio de ablación de las Iteraciones #2 a #5, xlm-roberta-large entrenado únicamente sobre datos reales demostró la mayor capacidad de generalización en Kaggle (0.89685), al no verse afectado por el ruido estilístico ni el suavizado de ironías que introducen las traducciones automáticas.

* Alineación exacta para el ensamble: Para poder combinar predicciones mediante votación suave (soft voting), era indispensable contar con las probabilidades calibradas de este modelo evaluadas sobre exactamente la misma partición val_df (seed=42) que el modelo de la Iteración #5 (val_probs_xlmr_large_bt_rdrop.npy).

###¿Qué otras alternativas había y por qué no se hicieron?

* Reutilizar únicamente las etiquetas discretas (submission_2_xlmr_large.csv) de la Iteración #2: Se descartó porque la votación mayoritaria dura (hard voting) descarta la incertidumbre de cada predicción y resulta inferior al soft voting cuando se combinan modelos bien calibrados.

* Entrenar xlm-roberta-large con batch_size = 16 sin acumulación ni gradient checkpointing: Descartado porque en entornos compartidos de Kaggle (T4 de 15 GB) provocaba errores OutOfMemoryError tras ejecutar previamente otros modelos en la misma sesión.

###¿Qué cambia frente al código anterior (Iteración #5)?

* Se elimina por completo el uso de trn_aug (datos traducidos por MarianMT), entrenando únicamente sobre trn_df (17,055 filas reales).

* Se sustituye la doble pasada estocástica con divergencia KL (R-Drop) por una pasada estándar con entropía cruzada directa (outputs.loss), lo que duplica la velocidad de entrenamiento por época y preserva la calibración pura del modelo base.

In [ ]:
!pip install -q sentencepiece sacremoses

import os
import re
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    MarianMTModel, MarianTokenizer, get_cosine_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Localizar automáticamente train.csv y eval.csv en /kaggle/input
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
eval_path = glob.glob('/kaggle/input/**/eval.csv', recursive=True)[0]
print("Rutas detectadas:\n -", train_path, "\n -", eval_path)

train_df = pd.read_csv(train_path, encoding='utf-8')
eval_df = pd.read_csv(eval_path, encoding='utf-8')

label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label = {0: 'negativo', 1: 'neutral', 2: 'positivo'}
train_df['target'] = train_df['label'].map(label2id)

def detect_lang(text):
    t = text.lower()
    words = set(re.findall(r'\b[a-záéíóúñãõçâêîôûàèù]+\b', t))
    es = len(words & {'el','la','los','las','que','en','un','una','por','para','con','pero','muy','lo','me','se','del','al','es','son','fue','está','más','también','cuando','todo','esta','este','bien','malo','bueno','llegó','compré','pedí','uso','días','casa'})
    en = len(words & {'the','and','to','of','is','in','it','for','that','with','on','was','but','not','are','have','this','my','very','good','great','bad','product','bought','use','days','quality','battery','would','after'})
    pt = len(words & {'não','com','uma','para','por','mais','como','mas','muito','bem','bom','boa','ruim','comprei','chegou','dias','casa','uso','também','quando','isso','essa','esse','são','está','fazer','qualidade','bateria','nas','dos','das','pelo','pela'})
    fr = len(words & {'le','la','les','des','est','une','dans','pour','pas','que','avec','sur','mais','plus','très','bien','bon','mauvais','acheté','livraison','jours','maison','qualité','batterie','depuis','tout','cette','sont','fait','avoir'})
    if es >= 3 and en >= 3:
        return 'es-en'
    scores = {'es': es, 'en': en, 'pt': pt, 'fr': fr}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

train_df['lang'] = train_df['text'].apply(detect_lang)
eval_df['lang'] = eval_df['text'].apply(detect_lang)

# Mismo split exacto (seed=42) para comparar limpio contra el baseline sin aumentación (0.9074 val / 0.89685 LB)
train_df['strat_key'] = train_df['label'] + '_' + train_df['lang']
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)
for train_idx, val_idx in skf.split(train_df, train_df['strat_key']):
    trn_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    break

def translate_batch(texts, model_name, batch_size=32):
    tok = MarianTokenizer.from_pretrained(model_name)
    mod = MarianMTModel.from_pretrained(model_name).to(device)
    mod.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(texts[i:i+batch_size], return_tensors='pt', padding=True, truncation=True, max_length=160).to(device)
            gen = mod.generate(**enc, max_length=160, num_beams=2)
            out.extend(tok.batch_decode(gen, skip_special_tokens=True))
    del mod, tok
    torch.cuda.empty_cache()
    return out

# Back-Translation Simétrico: conserva el idioma original y balancea 50% negativo / 50% positivo
def back_translate_balanced(df, src_lang, fwd_model, bwd_model, n_per_class=250):
    sub_neg = df[(df['lang'] == src_lang) & (df['label'] == 'negativo')].sample(n=n_per_class, random_state=42)
    sub_pos = df[(df['lang'] == src_lang) & (df['label'] == 'positivo')].sample(n=n_per_class, random_state=42)
    sub = pd.concat([sub_neg, sub_pos], ignore_index=True)

    pivot_texts = translate_batch(sub['text'].tolist(), fwd_model)
    bt_texts = translate_batch(pivot_texts, bwd_model)

    res = pd.DataFrame({
        'text': bt_texts,
        'label': sub['label'].values,
        'lang': src_lang,
        'aug_type': f'bt_{src_lang}'
    })
    res = res[res['text'].str.strip() != sub['text'].str.strip()].reset_index(drop=True)
    return res

print("Ejecutando Back-Translation EN -> ES -> EN (balanceado)...")
bt_en = back_translate_balanced(trn_df, 'en', 'Helsinki-NLP/opus-mt-en-es', 'Helsinki-NLP/opus-mt-es-en', n_per_class=250)

print("Ejecutando Back-Translation FR -> EN -> FR (balanceado)...")
bt_fr = back_translate_balanced(trn_df, 'fr', 'Helsinki-NLP/opus-mt-fr-en', 'Helsinki-NLP/opus-mt-en-fr', n_per_class=250)

print("Ejecutando Back-Translation ES -> EN -> ES sobre reseñas con concesiones (balanceado)...")
hard_es = trn_df[(trn_df['lang'] == 'es') & (trn_df['text'].str.lower().str.contains(r'\b(?:pero|aunque|sin embargo)\b', regex=True))]
bt_es = back_translate_balanced(hard_es, 'es', 'Helsinki-NLP/opus-mt-es-en', 'Helsinki-NLP/opus-mt-en-es', n_per_class=300)

trn_df['aug_type'] = 'original'
trn_aug = pd.concat([trn_df[['text', 'label', 'lang', 'aug_type']], bt_en, bt_fr, bt_es], ignore_index=True)
trn_aug['target'] = trn_aug['label'].map(label2id)
trn_aug = trn_aug.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("\nResumen datos aumentados por Back-Translation balanceado:")
print(trn_aug['aug_type'].value_counts())
print("Balance de clases en datos aumentados:")
print(trn_aug[trn_aug['aug_type'] != 'original']['label'].value_counts())
print(f"Total entrenamiento: {len(trn_df)} -> {len(trn_aug)} | Validación intacta: {len(val_df)}")

Dispositivo: cuda | GPU: Tesla T4
Rutas detectadas:
 - /kaggle/input/datasets/sebastianvargas21/data-p2/train.csv 
 - /kaggle/input/datasets/sebastianvargas21/data-p2/eval.csv


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


Ejecutando Back-Translation EN -> ES -> EN (balanceado)...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Ejecutando Back-Translation FR -> EN -> FR (balanceado)...


... (2 outputs left out)

Ejecutando Back-Translation ES -> EN -> ES sobre reseñas con concesiones (balanceado)...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


Resumen datos aumentados por Back-Translation balanceado:
aug_type
original    17055
bt_es         600
bt_fr         500
bt_en         499
Name: count, dtype: int64
Balance de clases en datos aumentados:
label
positivo    800
negativo    799
Name: count, dtype: int64
Total entrenamiento: 17055 -> 18654 | Validación intacta: 3411


Modelo xlm-roberta-large igual que en la iteración 5

In [ ]:
import gc, sys
sys.last_traceback = None
sys.last_value = None
for v in ['model', 'optimizer', 'scheduler', 'scaler', 'logits1', 'logits2', 'loss']:
    if v in globals():
        del globals()[v]
gc.collect()
torch.cuda.empty_cache()

MODEL_1 = 'xlm-roberta-large'
tok_1 = AutoTokenizer.from_pretrained(MODEL_1)

class CleanDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=160, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None
    def __len__(self): return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        item = {'input_ids': self.enc['input_ids'][idx], 'attention_mask': self.enc['attention_mask'][idx]}
        if self.labels is not None: item['labels'] = self.labels[idx]
        return item

trn_ds_1 = CleanDataset(trn_df['text'], tok_1, labels=trn_df['target'].values)
val_ds_1 = CleanDataset(val_df['text'], tok_1, labels=val_df['target'].values)
eval_ds_1 = CleanDataset(eval_df['text'], tok_1)

trn_loader_1 = DataLoader(trn_ds_1, batch_size=8, shuffle=True)
val_loader_1 = DataLoader(val_ds_1, batch_size=32, shuffle=False)
eval_loader_1 = DataLoader(eval_ds_1, batch_size=32, shuffle=False)

model_1 = AutoModelForSequenceClassification.from_pretrained(MODEL_1, num_labels=3)
model_1.gradient_checkpointing_enable()
model_1 = model_1.to(device)

opt_1 = torch.optim.AdamW(model_1.parameters(), lr=1e-5, weight_decay=0.01)
steps_1 = (len(trn_loader_1) // 2) * 4
sch_1 = get_cosine_schedule_with_warmup(opt_1, int(0.1 * steps_1), steps_1)
scaler_1 = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_acc_1 = 0.0
for epoch in range(4):
    model_1.train()
    opt_1.zero_grad()
    for step, batch in enumerate(trn_loader_1):
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = model_1(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device)
            ).loss / 2
        scaler_1.scale(loss).backward()
        if (step + 1) % 2 == 0 or (step + 1) == len(trn_loader_1):
            scaler_1.unscale_(opt_1)
            torch.nn.utils.clip_grad_norm_(model_1.parameters(), 1.0)
            scaler_1.step(opt_1)
            scaler_1.update()
            opt_1.zero_grad()
            sch_1.step()

    model_1.eval()
    v_probs = []
    with torch.no_grad():
        for batch in val_loader_1:
            logits = model_1(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
            v_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    v_probs = np.vstack(v_probs)
    acc = accuracy_score(val_df['target'], np.argmax(v_probs, axis=1))
    print(f"[XLM-R Sin Aug] Época {epoch+1}/4 | Val Acc: {acc:.4f}")
    if acc > best_acc_1:
        best_acc_1 = acc
        torch.save(model_1.state_dict(), '/kaggle/working/best_xlmr_sin_aug.pt')
        np.save('/kaggle/working/val_probs_xlmr_sin_aug.npy', v_probs)

model_1.load_state_dict(torch.load('/kaggle/working/best_xlmr_sin_aug.pt'))
model_1.eval()
e_probs_1 = []
with torch.no_grad():
    for batch in eval_loader_1:
        logits = model_1(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
        e_probs_1.append(torch.softmax(logits, dim=1).cpu().numpy())
np.save('/kaggle/working/eval_probs_xlmr_sin_aug.npy', np.vstack(e_probs_1))
print("Guardado Modelo 1 (XLM-R Sin Aug) exitosamente.")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_353/4228483072.py:58: UserWarning: Detected call of `lr_scheduler.step()` befo

[XLM-R Sin Aug] Época 1/4 | Val Acc: 0.8883
[XLM-R Sin Aug] Época 2/4 | Val Acc: 0.9038
[XLM-R Sin Aug] Época 3/4 | Val Acc: 0.9038
[XLM-R Sin Aug] Época 4/4 | Val Acc: 0.9062
Guardado Modelo 1 (XLM-R Sin Aug) exitosamente.


##2. Entrenamiento del Modelo 3 (microsoft/mdeberta-v3-base) para Diversidad Arquitectónica
###¿Qué se hizo?

* Se liberó de memoria el Modelo 1 (del model_1, torch.cuda.empty_cache()) y se cargó una segunda familia de arquitectura completamente distinta: microsoft/mdeberta-v3-base.

* El modelo se convirtió explícitamente a precisión simple de 32 bits (.float()) y se entrenó en fp32 nativo (sin autocast ni GradScaler) durante 4 épocas sobre trn_df (batch_size = 16, LR = 2e-5, weight_decay = 0.01, scheduler coseno con 10% de warmup).

* Se evaluó sobre el mismo val_df (alcanzando Val Accuracy = 0.9035 en la época 4) y se exportaron sus matrices de probabilidad en validación (val_probs_mdeberta_v3.npy) y evaluación (eval_probs_mdeberta_v3.npy).

###¿Por qué se decidió hacer eso?

* Maximizar la diversidad del ensamble mediante familias de modelos distintas: Un ensamble solo supera consistentemente a sus componentes individuales cuando los modelos cometen errores no correlacionados. Mientras XLM-RoBERTa utiliza codificación posicional absoluta y preentrenamiento Masked Language Modeling (MLM), mDeBERTa-v3 emplea atención relativa desenredada (disentangled relative positional attention) y preentrenamiento discriminativo tipo ELECTRA (Replaced Token Detection), capturando de forma distinta las relaciones de distancia entre negaciones, concesiones (pero, aunque) y el veredicto final.

* Estabilidad numérica en fp32: Los pesos preentrenados de microsoft/mdeberta-v3-base en HuggingFace se distribuyen en fp16 y su mecanismo de atención relativa sufre desbordamientos (overflow) o errores de escalado (ValueError: Attempting to unscale FP16 gradients) bajo precisión mixta clásica; forzar .float() garantizó una convergencia limpia y estable.

###¿Qué otras alternativas había y por qué no se hicieron?

* Combinar únicamente variantes de xlm-roberta-large (por ejemplo, Iteración #2 + Iteración #5): Aunque reduce ligeramente la varianza, ambos modelos comparten exactamente el mismo tokenizador (SentencePiece), la misma arquitectura de 24 capas y los mismos pesos iniciales preentrenados, limitando la ganancia por diversidad.

* Usar bert-base-multilingual-cased (mBERT) o distilbert-base-multilingual-cased: Se descartaron porque tienen menor capacidad representacional que mDeBERTa-v3 en tareas multilingües complejas con ironía y code-switching.

* Entrenar mDeBERTa-v3 en bfloat16: No estaba soportado de forma nativa por las GPUs Tesla T4 asignadas en Kaggle (arquitectura Turing), por lo que fp32 era la opción más segura y rápida.

###¿Qué cambia frente al código anterior (Celda A / Iteración #5)?

* Cambia por completo el backbone neuronal (DebertaV2ForSequenceClassification en lugar de XLMRobertaForSequenceClassification), su tokenizador y su tasa de aprendizaje (2e-5, adecuada para arquitecturas de tamaño base frente al 1e-5 requerido por modelos large).


In [ ]:
import gc
for v in ['model_3', 'opt_3', 'sch_3', 'scaler_3']:
    if v in globals():
        del globals()[v]
gc.collect()
torch.cuda.empty_cache()

MODEL_3 = 'microsoft/mdeberta-v3-base'
tok_3 = AutoTokenizer.from_pretrained(MODEL_3)

trn_ds_3 = CleanDataset(trn_df['text'], tok_3, labels=trn_df['target'].values)
val_ds_3 = CleanDataset(val_df['text'], tok_3, labels=val_df['target'].values)
eval_ds_3 = CleanDataset(eval_df['text'], tok_3)

trn_loader_3 = DataLoader(trn_ds_3, batch_size=16, shuffle=True)
val_loader_3 = DataLoader(val_ds_3, batch_size=32, shuffle=False)
eval_loader_3 = DataLoader(eval_ds_3, batch_size=32, shuffle=False)

# Forzar float32 (.float()) evita el error de GradScaler y previene overflow en DeBERTa-v3
model_3 = AutoModelForSequenceClassification.from_pretrained(MODEL_3, num_labels=3).float().to(device)
opt_3 = torch.optim.AdamW(model_3.parameters(), lr=2e-5, weight_decay=0.01)
steps_3 = len(trn_loader_3) * 4
sch_3 = get_cosine_schedule_with_warmup(opt_3, int(0.1 * steps_3), steps_3)

best_acc_3 = 0.0
for epoch in range(4):
    model_3.train()
    for batch in trn_loader_3:
        opt_3.zero_grad()
        loss = model_3(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['labels'].to(device)
        ).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_3.parameters(), 1.0)
        opt_3.step()
        sch_3.step()

    model_3.eval()
    v_probs = []
    with torch.no_grad():
        for batch in val_loader_3:
            logits = model_3(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
            v_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    v_probs = np.vstack(v_probs)
    acc = accuracy_score(val_df['target'], np.argmax(v_probs, axis=1))
    print(f"[mDeBERTa-v3] Época {epoch+1}/4 | Val Acc: {acc:.4f}")
    if acc > best_acc_3:
        best_acc_3 = acc
        torch.save(model_3.state_dict(), '/kaggle/working/best_mdeberta_v3.pt')
        np.save('/kaggle/working/val_probs_mdeberta_v3.npy', v_probs)

model_3.load_state_dict(torch.load('/kaggle/working/best_mdeberta_v3.pt'))
model_3.eval()
e_probs_3 = []
with torch.no_grad():
    for batch in eval_loader_3:
        logits = model_3(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
        e_probs_3.append(torch.softmax(logits, dim=1).cpu().numpy())
np.save('/kaggle/working/eval_probs_mdeberta_v3.npy', np.vstack(e_probs_3))
print("Guardado Modelo 3 (mDeBERTa-v3) exitosamente.")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight 

[mDeBERTa-v3] Época 1/4 | Val Acc: 0.8877
[mDeBERTa-v3] Época 2/4 | Val Acc: 0.9009
[mDeBERTa-v3] Época 3/4 | Val Acc: 0.9021
[mDeBERTa-v3] Época 4/4 | Val Acc: 0.9035
Guardado Modelo 3 (mDeBERTa-v3) exitosamente.


##3. Optimización de Pesos del Ensamble (Soft Voting) e Inferencia Final sobre eval.csv
###¿Qué se hizo?
* Se cargaron las tres matrices de probabilidad de validación alineadas (3,411 × 3):p_val_1: xlm-roberta-large sin aumentación (0.9062 Val Acc).p_val_2: xlm-roberta-large con Back-Translation balanceado + R-Drop de la Iteración #5 (0.9074 Val Acc). p_val_3: microsoft/mdeberta-v3-base sin aumentación (0.9035 Val Acc).
* Se ejecutó una búsqueda sistemática en grilla (grid search) sobre el simplex de pesos $w_1 + w_2 + w_3 = 1$ para maximizar el accuracy en validación, hallando la combinación óptima: $w_1 = 0.30$ (XLM-R Sin Aug), $w_2 = 0.25$ (XLM-R BT + R-Drop) y $w_3 = 0.45$ (mDeBERTa-v3), que elevó el desempeño en validación a 0.9112 (Macro F1 = 0.9184).
* Se aplicó esa misma combinación lineal ponderada sobre las matrices de probabilidad de eval.csv (6,781 × 3), exportando el archivo submission_6_ensemble.csv y verificando el equilibrio de clases predichas (positivo: 43.80%, negativo: 37.09%, neutral: 19.11%).
###¿Por qué se decidió hacer eso?
* Reducción de varianza y corrección mutua de errores: El hecho de que el optimizador asignara el mayor peso (0.45) a mDeBERTa-v3 (a pesar de tener el menor accuracy individual de los tres, 0.9035) frente al peso combinado de 0.55 para la familia XLM-RoBERTa-large (0.30 + 0.25) demuestra empíricamente la complementariedad entre ambas arquitecturas: mDeBERTa-v3 corrige casos donde XLM-RoBERTa falla y viceversa.
* Aprovechamiento de la regularización de la Iteración #5 sin sufrir su ruido: Integrar p_val_2 con un peso moderado (0.25) aporta la robustez en concesiones adquirida mediante Back-Translation + R-Drop (0.9099 en casos adversativos), mientras que el 0.75 restante de modelos entrenados sobre datos 100% reales mantiene anclada la distribución lingüística natural.
###¿Qué otras alternativas había y por qué no se hicieron?
* Promedio simple equiponderado ($w_1 = w_2 = w_3 = 1/3$): Se descartó porque daría dos tercios (66.7%) del voto a la misma familia arquitectónica (XLM-RoBERTa-large) y solo un tercio (33.3%) a mDeBERTa-v3, subutilizando la diversidad estructural.
* Hard Voting sobre las clases predichas: Descartada porque pierde la información de confianza de cada modelo (por ejemplo, cuando un modelo predice con 0.51 vs 0.49 y otro predice con 0.98 vs 0.02).
* Stacking con meta-clasificador (Regresión Logística o XGBoost sobre los logits): Se evitó para no introducir riesgo de sobreajuste sobre las 3,411 muestras de validación; una combinación convexa de 3 pesos ($w_i \ge 0, \sum w_i = 1$) tiene apenas 2 grados de libertad y generaliza de forma mucho más estable.
###¿Qué cambia frente al código anterior (Iteración #5)?
Se pasa de inferencia con un único modelo neuronal a un sistema de ensamble heterogéneo multi-arquitectura mediante combinación lineal de distribuciones softmax optimizada por búsqueda en grilla.

In [ ]:
p_val_1 = np.load('/kaggle/working/val_probs_xlmr_sin_aug.npy')         # XLM-R Large Sin Aug
p_val_2 = np.load('/kaggle/working/val_probs_xlmr_large_bt_rdrop.npy')  # XLM-R Large BT + R-Drop (Iter 5)
p_val_3 = np.load('/kaggle/working/val_probs_mdeberta_v3.npy')          # mDeBERTa-v3 Base Sin Aug

y_true = val_df['target'].values

print("=== Desempeño Individual en Validación ===")
print(f"1. XLM-R Large (Sin Aug):     {accuracy_score(y_true, np.argmax(p_val_1, axis=1)):.4f}")
print(f"2. XLM-R Large (BT + R-Drop): {accuracy_score(y_true, np.argmax(p_val_2, axis=1)):.4f}")
print(f"3. mDeBERTa-v3 (Sin Aug):     {accuracy_score(y_true, np.argmax(p_val_3, axis=1)):.4f}")

best_w = (0.45, 0.30, 0.25)
best_ens_acc = 0.0

for w1 in np.linspace(0.3, 0.6, 7):
    for w2 in np.linspace(0.15, 0.45, 7):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 < 0.05 or w3 > 0.5:
            continue
        p_blend = w1 * p_val_1 + w2 * p_val_2 + w3 * p_val_3
        acc = accuracy_score(y_true, np.argmax(p_blend, axis=1))
        if acc > best_ens_acc:
            best_ens_acc = acc
            best_w = (round(w1, 2), round(w2, 2), round(w3, 2))

w1, w2, w3 = best_w
val_ens_probs = w1 * p_val_1 + w2 * p_val_2 + w3 * p_val_3
val_ens_preds = np.argmax(val_ens_probs, axis=1)

print(f"\n=== Ensamble Soft Voting Óptimo (Pesos: XLM-R={w1}, BT+RDrop={w2}, mDeBERTa={w3}) ===")
print(f"Val Accuracy Ensamble: {best_ens_acc:.4f}")
print(classification_report(y_true, val_ens_preds, target_names=['negativo', 'neutral', 'positivo'], digits=4))

p_eval_1 = np.load('/kaggle/working/eval_probs_xlmr_sin_aug.npy')
p_eval_2 = np.load('/kaggle/working/eval_probs_xlmr_large_bt_rdrop.npy')
p_eval_3 = np.load('/kaggle/working/eval_probs_mdeberta_v3.npy')

eval_ens_probs = w1 * p_eval_1 + w2 * p_eval_2 + w3 * p_eval_3
eval_ens_preds = np.argmax(eval_ens_probs, axis=1)

submission_6 = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_ens_preds]
})
submission_6.to_csv('/kaggle/working/submission_6_ensemble.csv', index=False)

print("\nArchivo guardado en /kaggle/working/submission_6_ensemble.csv")
print("Distribución de predicciones en eval.csv:")
print(submission_6['answer'].value_counts(normalize=True).round(4))

=== Desempeño Individual en Validación ===
1. XLM-R Large (Sin Aug):     0.9062
2. XLM-R Large (BT + R-Drop): 0.9074
3. mDeBERTa-v3 (Sin Aug):     0.9035

=== Ensamble Soft Voting Óptimo (Pesos: XLM-R=0.3, BT+RDrop=0.25, mDeBERTa=0.45) ===
Val Accuracy Ensamble: 0.9112
              precision    recall  f1-score   support

    negativo     0.8828    0.8699    0.8763      1230
     neutral     1.0000    0.9979    0.9989       932
    positivo     0.8731    0.8871    0.8801      1249

    accuracy                         0.9112      3411
   macro avg     0.9187    0.9183    0.9184      3411
weighted avg     0.9113    0.9112    0.9112      3411


Archivo guardado en /kaggle/working/submission_6_ensemble.csv
Distribución de predicciones en eval.csv:
answer
positivo    0.4380
negativo    0.3709
neutral     0.1911
Name: proportion, dtype: float64


En validación, el ensamble superó a los tres modelos individuales (0.9062, 0.9074 y 0.9035 $\rightarrow$ 0.9112), y en el leaderboard público de Kaggle alcanzó nuestro mejor puntaje hasta el momento